# 07 — Results Visualization and Consistency Checks
This notebook builds consolidated figures/tables from generated artifacts and verifies plotted metrics match recomputed values exactly.

Coverage: equity curves, drawdowns, Sharpe/Sortino/volatility, action sequences, and anomaly flags.

In [ ]:
from __future__ import annotations

from pathlib import Path
import random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display

from src.utils.config_loader import resolve_config
from src.utils.seed import set_global_seed
from src.utils.artifact_manager import ArtifactManager
from src.workflows.data_workflow import run_data_workflow
from src.workflows.train_workflow import run_train_workflow
from src.workflows.evaluate_workflow import run_evaluate_workflow
from src.evaluation.metrics import compute_metrics

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'src').exists() and (candidate / 'configs').exists():
            return candidate
    raise FileNotFoundError('Repository root not found.')

ROOT = find_repo_root(Path.cwd())
CONFIG = resolve_config(root=str(ROOT), agent_config='configs/agents/dqn.yaml')
SEED = int(CONFIG.get('training', {}).get('random_seed', 42))
set_global_seed(SEED, deterministic_torch=True)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

PAIR = CONFIG['data']['pairs'][0]
OUTPUTS_ROOT = str(ROOT / 'outputs')
am = ArtifactManager(OUTPUTS_ROOT)
RUN_TAG = 'notebook_results_viz'

## Ensure artifact availability
If required artifacts are missing, we generate a lightweight deterministic run so this notebook remains executable end-to-end.

In [ ]:
run_dir = am.agent_result_dir('dqn', PAIR) / RUN_TAG
metric_equity_path = run_dir / 'metrics' / 'test' / 'equity_curve.csv'

if not metric_equity_path.exists():
    cfg = resolve_config(
        root=str(ROOT),
        agent_config='configs/agents/dqn.yaml',
        cli_overrides={
            'training': {'total_timesteps': 3000, 'warmup_steps': 300, 'checkpoint_interval': 1000, 'resume': {'enabled': False}},
            'agent': {'training': {'learn_start_steps': 300, 'learn_frequency': 2}},
        },
    )
    data = run_data_workflow(cfg, pairs=[PAIR], root=str(ROOT))
    train_df, test_df = data[PAIR]['train'], data[PAIR]['test']
    _ = run_train_workflow(cfg, PAIR, train_df, outputs_root=OUTPUTS_ROOT, run_tag=RUN_TAG)
    _ = run_evaluate_workflow(
        config=cfg,
        pair=PAIR,
        eval_df=test_df,
        run_dir=run_dir,
        checkpoint_path=str(run_dir / 'checkpoints' / 'checkpoint_latest.pt'),
        split_name='test',
    )

assert metric_equity_path.exists(), f'Expected equity artifact not found: {metric_equity_path}'
print(f'Using run directory: {run_dir}')

## Equity and drawdown curves

In [ ]:
equity_df = pd.read_csv(run_dir / 'metrics' / 'test' / 'equity_curve.csv')
drawdown_df = pd.read_csv(run_dir / 'metrics' / 'test' / 'drawdown_curve.csv')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(equity_df['timestep'], equity_df['equity_value'])
axes[0].set_title('Equity curve (test)')
axes[0].set_xlabel('timestep')
axes[0].set_ylabel('equity')

axes[1].plot(drawdown_df['timestep'], drawdown_df['drawdown_value'], color='darkred')
axes[1].set_title('Drawdown curve (test)')
axes[1].set_xlabel('timestep')
axes[1].set_ylabel('drawdown')

plt.tight_layout()
plt.show()

## Performance metric reconciliation
We recompute metrics from the equity curve and confirm they match snapshot CSV values used in reports and plots.

In [ ]:
equity = equity_df['equity_value'].to_numpy(dtype=float)
metrics_recomputed = compute_metrics(
    equity_curve=equity,
    trade_log=None,
    periods_per_year=CONFIG['evaluation']['periods_per_year'],
    risk_free_rate=CONFIG['evaluation']['risk_free_rate'],
)

snapshot = {}
for metric_name in ['cumulative_return', 'sharpe_ratio', 'max_drawdown', 'turnover']:
    p = run_dir / 'metrics' / 'test' / f'{metric_name}.csv'
    if p.exists():
        snapshot[metric_name] = float(pd.read_csv(p)['value'].iloc[0])

rows = []
for k, v in snapshot.items():
    recomputed = float(metrics_recomputed[k])
    delta = abs(recomputed - v)
    rows.append({'metric': k, 'snapshot': v, 'recomputed': recomputed, 'abs_delta': delta})
    assert delta < 1e-8, f'Mismatch for {k}: {delta}'

metric_check_df = pd.DataFrame(rows)
display(metric_check_df)

## Agent action sequence analysis

In [ ]:
actions_path = run_dir / 'metrics' / 'test' / 'actions_sequence.csv'
actions_df = pd.read_csv(actions_path) if actions_path.exists() else pd.DataFrame()

if not actions_df.empty:
    action_counts = actions_df['action_name'].value_counts(dropna=False).rename_axis('action_name').reset_index(name='count')
    display(action_counts.head(20))

    plt.figure(figsize=(10, 4))
    action_counts.plot(kind='bar', x='action_name', y='count', legend=False, title='Action frequency (test)')
    plt.tight_layout()
    plt.show()
else:
    print(f'No actions sequence found at {actions_path}')

## Save consolidated visualization artifacts and anomaly notes

In [ ]:
summary_dir = Path(OUTPUTS_ROOT) / 'results' / 'visualization_summary' / PAIR
summary_dir.mkdir(parents=True, exist_ok=True)

# Save combined metrics table
combined_metrics = metric_check_df.copy()
combined_metrics.to_csv(summary_dir / 'metrics_reconciliation.csv', index=False)

# Save equity + drawdown merged table
merged_curve = equity_df.merge(drawdown_df, on='timestep', how='inner')
merged_curve.to_csv(summary_dir / 'equity_drawdown_series.csv', index=False)

anomalies = []
if not metric_check_df.empty and (metric_check_df['abs_delta'] > 1e-10).any():
    anomalies.append('Metric reconciliation has non-zero deltas above strict threshold.')
if not actions_df.empty and actions_df['was_legal'].astype(str).str.lower().eq('false').mean() > 0.05:
    anomalies.append('Illegal-action incidence exceeds 5% in action sequence.')

notes_df = pd.DataFrame({'note': anomalies or ['No major anomalies detected.']})
notes_df.to_csv(summary_dir / 'analysis_notes.csv', index=False)

print(f'✅ Saved consolidated tables to {summary_dir}')
display(notes_df)